In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
path_to_class_folders="/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat"

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from torch.cuda.amp import autocast, GradScaler

In [ ]:
ROOT_DIR = "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat"

SUBFOLDER = "In_focus"

In [ ]:
SELECTED_CLASSES = [
    "Thalassiosira sp",
    "Diatom 4 (c. concavicornus)",
    "Diatom 3 (ditylum sp.)",
    "Diatom 2",
    "Diatom 1 (c. debilis)",
    "Copepod Nauplii",
    "Copepod",
    "Ciliate",
    "Ceratium muelleri (singular)",
    "Ceratium furca (singular)"
]

In [ ]:
CLASS_PATHS = [
    os.path.join(ROOT_DIR, cls)
    for cls in SELECTED_CLASSES
]

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class PlanktonDataset(Dataset):

    def __init__(self,
                 class_paths,
                 subfolder="In_focus",
                 transform=None):

        self.transform = transform
        self.samples = []

        #################################################
        # Label Mapping
        #################################################

        self.class_names = [
            os.path.basename(path)
            for path in class_paths
        ]

        self.class_to_idx = {
            cls: idx
            for idx, cls in enumerate(self.class_names)
        }

        #################################################
        # Read Images
        #################################################

        valid_extensions = (".tif", ".tiff")

        for class_path in class_paths:

            class_name = os.path.basename(class_path)

            label = self.class_to_idx[class_name]

            image_folder = os.path.join(class_path,
                                        subfolder)

            if not os.path.exists(image_folder):

                print(f"Warning : {image_folder} not found")

                continue

            for image in sorted(os.listdir(image_folder)):

                if image.lower().endswith(valid_extensions):

                    image_path = os.path.join(image_folder,
                                              image)

                    self.samples.append(
                        (image_path,
                         label)
                    )

        print("="*50)
        print("Selected Classes :", len(self.class_names))
        print("Total Images     :", len(self.samples))
        print("="*50)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        image_path, label = self.samples[index]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [ ]:
from torchvision import transforms

IMG_SIZE = 128
# Validation/Test transformations
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
full_dataset = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=None
)

print("Total images:", len(full_dataset))
print("Classes:", full_dataset.class_names)
print("Number of classes:", len(full_dataset.class_names))

In [ ]:
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Extract labels directly from the full dataset
# Keep them as normal Python integers
# --------------------------------------------------

dataset_labels = [
    int(label)
    for _, label in full_dataset.samples
]

# All dataset indices
indices = list(range(len(full_dataset)))

print("Number of images :", len(indices))
print("Number of labels :", len(dataset_labels))

# --------------------------------------------------
# 70% Train / 30% Temporary
# --------------------------------------------------

train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=dataset_labels
)

print("Train :", len(train_indices))
print("Temp  :", len(temp_indices))

In [ ]:
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# Extract labels directly from the full dataset
# Keep them as normal Python integers
# --------------------------------------------------

dataset_labels = [
    int(label)
    for _, label in full_dataset.samples
]

# All dataset indices
indices = list(range(len(full_dataset)))

print("Number of images :", len(indices))
print("Number of labels :", len(dataset_labels))

# --------------------------------------------------
# 70% Train / 30% Temporary
# --------------------------------------------------

train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.30,
    random_state=42,
    stratify=dataset_labels
)

print("Train :", len(train_indices))
print("Temp  :", len(temp_indices))

In [ ]:
# Get labels corresponding only to temp_indices
temp_labels = [
    dataset_labels[i]
    for i in temp_indices
]

# Split the remaining 30% into:
# 15% validation
# 15% test
val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("\nFinal Split")
print("=" * 40)
print("Train      :", len(train_indices))
print("Validation :", len(val_indices))
print("Test       :", len(test_indices))

In [ ]:
dataset = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=train_transform
)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

In [ ]:
import torch
import torch.nn as nn
class DIBB(nn.Module):
    """
    Depthwise Inverted Bottleneck Block

    Input
      ↓
    1x1 Expansion
      ↓
    3x3 Depthwise Convolution
      ↓
    1x1 Projection
      ↓
    Residual Addition
    """

    def __init__(self, in_channels, out_channels,
                 expansion_ratio=2, stride=1):

        super().__init__()

        hidden_channels = int(in_channels * expansion_ratio)

        # 1. Expansion
        self.expand = nn.Sequential(
            nn.Conv2d(
                in_channels,
                hidden_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU6(inplace=True)
        )

        # 2. Depthwise convolution
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                hidden_channels,
                hidden_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=hidden_channels,
                bias=False
            ),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU6(inplace=True)
        )

        # 3. Projection
        self.project = nn.Sequential(
            nn.Conv2d(
                hidden_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

        # Residual can only be used when shapes match
        self.use_residual = (
            stride == 1 and
            in_channels == out_channels
        )

    def forward(self, x):

        identity = x

        x = self.expand(x)
        x = self.depthwise(x)
        x = self.project(x)

        if self.use_residual:
            x = x + identity

        return x

In [ ]:
x = torch.randn(8, 32, 128, 128)

block = DIBB(
    in_channels=32,
    out_channels=32,
    expansion_ratio=2
)

y = block(x)

print("Input :", x.shape)
print("Output:", y.shape)

In [ ]:
def make_dibb_stage(
    in_channels,
    out_channels,
    num_blocks,
    expansion_ratio=2,
    first_stride=1
):

    layers = []

    # First block may change channels/resolution
    layers.append(
        DIBB(
            in_channels=in_channels,
            out_channels=out_channels,
            expansion_ratio=expansion_ratio,
            stride=first_stride
        )
    )

    # Remaining blocks preserve shape
    for _ in range(1, num_blocks):

        layers.append(
            DIBB(
                in_channels=out_channels,
                out_channels=out_channels,
                expansion_ratio=expansion_ratio,
                stride=1
            )
        )

    return nn.Sequential(*layers)

In [ ]:
class PlanktonDIBBNet(nn.Module):

    def __init__(self, num_classes=10):
        super().__init__()

        # =====================================
        # Initial convolution
        # =====================================

        self.stem = nn.Sequential(

            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(32),

            nn.ReLU6(inplace=True)
        )

        # =====================================
        # DIBB × 2
        # =====================================

        self.stage1 = make_dibb_stage(
            in_channels=32,
            out_channels=32,
            num_blocks=2,
            expansion_ratio=2,
            first_stride=1
        )

        # =====================================
        # DIBB × 3
        # =====================================

        self.stage2 = make_dibb_stage(
            in_channels=32,
            out_channels=64,
            num_blocks=3,
            expansion_ratio=2,
            first_stride=2
        )

        # =====================================
        # DIBB × 2
        # =====================================

        self.stage3 = make_dibb_stage(
            in_channels=64,
            out_channels=128,
            num_blocks=2,
            expansion_ratio=2,
            first_stride=2
        )

        # =====================================
        # DIBB × 1
        # =====================================

        self.stage4 = make_dibb_stage(
            in_channels=128,
            out_channels=256,
            num_blocks=1,
            expansion_ratio=2,
            first_stride=2
        )

        # =====================================
        # Global Average Pooling
        # =====================================

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # =====================================
        # Classifier
        # =====================================

        self.classifier = nn.Linear(
            256,
            num_classes
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.stage1(x)

        x = self.stage2(x)

        x = self.stage3(x)

        x = self.stage4(x)

        x = self.avgpool(x)

        # B × 256 × 1 × 1
        #          ↓
        # B × 256

        x = torch.flatten(x, 1)

        # Raw logits
        x = self.classifier(x)

        return x

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = PlanktonDIBBNet(
    num_classes=10
).to(DEVICE)

print(model)

In [ ]:
x = torch.randn(
    8,
    3,
    128,
    128
).to(DEVICE)

outputs = model(x)

print("Input shape :", x.shape)
print("Output shape:", outputs.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # ----------------------------
        # Clear old gradients
        # ----------------------------

        optimizer.zero_grad()

        # ----------------------------
        # Forward propagation
        # ----------------------------

        outputs = model(images)

        # ----------------------------
        # Calculate loss
        # ----------------------------

        loss = criterion(outputs, labels)

        # ----------------------------
        # Backpropagation
        # ----------------------------

        loss.backward()

        # ----------------------------
        # Update weights
        # ----------------------------

        optimizer.step()

        # ----------------------------
        # Statistics
        # ----------------------------

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    epoch_loss = running_loss / len(train_loader)

    epoch_accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

In [ ]:
test_base = PlanktonDataset(
    class_paths=CLASS_PATHS,
    subfolder="In_focus",
    transform=test_transform
)

In [ ]:
from torch.utils.data import Subset

test_dataset = Subset(
    test_base,
    test_indices
)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
print("Number of test images :", len(test_dataset))
print("Number of test batches:", len(test_loader))

images, labels = next(iter(test_loader))

print("\nFirst Test Batch")
print("=" * 40)

print("Image batch shape :", images.shape)
print("Label batch shape :", labels.shape)
print("Labels            :", labels)

In [ ]:
import torch

# ============================================================
# TEST / INFERENCE
# ============================================================

model.eval()

correct = 0
total = 0
test_loss = 0.0


# Disable gradient calculation during inference
with torch.no_grad():

    for images, labels in test_loader:

        # Move data to GPU/CPU
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        outputs = model(images)

        # ----------------------------------------------------
        # Calculate test loss
        # ----------------------------------------------------

        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        # ----------------------------------------------------
        # Get predicted class
        # ----------------------------------------------------

        _, predicted = torch.max(outputs, dim=1)

        # ----------------------------------------------------
        # Calculate number of correct predictions
        # ----------------------------------------------------

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


# ============================================================
# FINAL RESULTS
# ============================================================

average_test_loss = test_loss / total

test_accuracy = 100.0 * correct / total


print("=" * 50)
print("TEST DATASET RESULTS")
print("=" * 50)

print(f"Total Test Images     : {total}")
print(f"Correct Predictions   : {correct}")
print(f"Incorrect Predictions : {total - correct}")

print("-" * 50)

print(f"Test Loss             : {average_test_loss:.4f}")
print(f"Test Accuracy         : {test_accuracy:.2f}%")

print("=" * 50)

In [ ]:
pip install thop

In [ ]:
from thop import profile
import torch

# Get one batch from test_loader
images, labels = next(iter(test_loader))

print("Test batch shape:", images.shape)

# Select ONE image
test_image = images[0].unsqueeze(0).to(DEVICE)

print("Selected image shape:", test_image.shape)

# Put model in evaluation mode
model.eval()

# Calculate MACs and parameters
macs, params = profile(
    model,
    inputs=(test_image,),
    verbose=False
)

# Convert units
gmacs = macs / 1e9

# Convention: 1 MAC ≈ 2 FLOPs
gflops = (2 * macs) / 1e9

million_params = params / 1e6

print("\n" + "=" * 50)
print("DIBB MODEL COMPUTATIONAL COMPLEXITY")
print("=" * 50)

print(f"Input Shape : {tuple(test_image.shape)}")
print(f"Parameters  : {million_params:.4f} M")
print(f"MACs        : {macs / 1e6:.2f} M")
print(f"GMACs       : {gmacs:.4f}")
print(f"GFLOPs      : {gflops:.4f}")

print("=" * 50)